# 🎨 Notebook 1 · Online Stock Brokerage — Class Design

## 🛠️ Setup

```bash
cd 07-object-oriented-design/online-stock-brokerage
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🎯 What we're modeling

An **online stock brokerage** lets users deposit cash, place buy/sell orders on stocks, and track a portfolio.

We'll build the **class design** step by step: first a naïve version full of `if/elif`, then refactor towards clean OO using inheritance and a small set of well-defined classes.

If you're brand new: think of a class as a **template for a thing** (a `Stock`, an `Account`, an `Order`). An object is one concrete copy made from that template.


## 1️⃣ What does a brokerage actually do?

Let's list the **nouns** in plain English — those usually become classes.

- A **stock** has a symbol (e.g. `"AAPL"`) and a current price.
- An **account** belongs to a user; it has cash and a portfolio.
- A **portfolio** holds positions (how many shares of each stock you own).
- An **order** is a request to buy or sell some quantity of a stock.
  - It can be a **market**, **limit**, or **stop** order.
- A **trade** is the record of an order that actually executed.
- An **exchange** matches orders and produces trades.

Drawing this as a diagram:

```
 Account ─◆─ Portfolio ─◆─ Position(symbol, qty, avg_price)
    │
    │ places
    ▼
  Order ──▶ Stock
    │
    │ fills into
    ▼
  Trade
```

The ◆ means "owns" (composition). The arrow `──▶` means "refers to".


## 2️⃣ Attempt 0 — one giant function with `if/elif` (bad 🔴)

A beginner might skip classes entirely and just write one function that handles every order type with branches.


In [ ]:
from dataclasses import dataclass

@dataclass
class Stock:
    symbol: str
    price: float

def try_fill_bad(order: dict, market_price: float) -> dict | None:
    """Returns a 'trade' dict if the order fills, else None."""
    kind = order["kind"]

    if kind == "market":
        fill_price = market_price

    elif kind == "limit":
        if order["side"] == "buy" and market_price > order["limit"]:
            return None
        if order["side"] == "sell" and market_price < order["limit"]:
            return None
        fill_price = market_price

    elif kind == "stop":
        if order["side"] == "buy" and market_price < order["stop"]:
            return None
        if order["side"] == "sell" and market_price > order["stop"]:
            return None
        fill_price = market_price

    else:
        raise ValueError(f"unknown kind: {kind}")

    return {"symbol": order["symbol"], "qty": order["qty"], "price": fill_price}

# demo
print(try_fill_bad({"kind":"market","side":"buy","symbol":"AAPL","qty":10}, market_price=180))
print(try_fill_bad({"kind":"limit","side":"buy","symbol":"AAPL","qty":10,"limit":170}, market_price=168))
print(try_fill_bad({"kind":"limit","side":"buy","symbol":"AAPL","qty":10,"limit":170}, market_price=175))


### What's wrong with this? 🔴

- **Every new order type (trailing stop, iceberg, …) means editing `try_fill_bad`** — a classic Open/Closed Principle violation.
- Orders are **dicts with magic string keys** — no autocomplete, no type checking, easy to typo.
- The **status lifecycle** (pending → filled / cancelled) lives nowhere.
- Adding validation (e.g. `qty > 0`) means yet more branches.
- Hard to unit-test any single order type in isolation.


## 3️⃣ Attempt 1 — give each concept a class (better 🟡)

Let's start by making the nouns into real classes. We'll use `dataclass` so we get `__init__` and a nice `repr` for free.


In [ ]:
from enum import Enum
from dataclasses import dataclass, field

class Side(Enum):
    BUY = "buy"
    SELL = "sell"

class OrderStatus(Enum):
    PENDING = "pending"
    FILLED = "filled"
    CANCELLED = "cancelled"

@dataclass
class Position:
    symbol: str
    qty: int = 0
    avg_price: float = 0.0   # weighted average cost - useful for P&L later

@dataclass
class Portfolio:
    positions: dict[str, Position] = field(default_factory=dict)

@dataclass
class Account:
    id: int
    cash: float
    portfolio: Portfolio = field(default_factory=Portfolio)

alice = Account(id=1, cash=10_000)
print(alice)


This already feels cleaner — `alice.cash`, `alice.portfolio.positions` are self-documenting. But we still haven't modeled orders properly.


## 4️⃣ Attempt 2 — **polymorphism** for order types (best 🟢)

Here's the key OO move: instead of one function branching on `kind`, each order type is its **own class** that knows how to answer two simple questions:

1. `can_fill(market_price)` → at this price, should I execute?
2. `fill_price(market_price)` → if I do, at what price?

This is the **Strategy pattern** in disguise: the varying behavior per order type is encapsulated behind a common interface.


In [ ]:
from abc import ABC, abstractmethod

class Order(ABC):
    def __init__(self, account: Account, stock: Stock, side: Side, qty: int):
        if qty <= 0:
            raise ValueError("qty must be positive")
        self.account = account
        self.stock = stock
        self.side = side
        self.qty = qty
        self.status = OrderStatus.PENDING

    @abstractmethod
    def can_fill(self, market_price: float) -> bool: ...

    @abstractmethod
    def fill_price(self, market_price: float) -> float: ...

    def __repr__(self):
        return f"{type(self).__name__}({self.side.value} {self.qty} {self.stock.symbol}, {self.status.value})"


class MarketOrder(Order):
    """Execute immediately at whatever the market price is."""
    def can_fill(self, market_price): return True
    def fill_price(self, market_price): return market_price


class LimitOrder(Order):
    """Execute only if the price is at least as good as my limit."""
    def __init__(self, account, stock, side, qty, limit_price):
        super().__init__(account, stock, side, qty)
        self.limit_price = limit_price
    def can_fill(self, market_price):
        if self.side == Side.BUY:  return market_price <= self.limit_price
        else:                      return market_price >= self.limit_price
    def fill_price(self, market_price): return market_price


class StopOrder(Order):
    """Becomes a market order once the price crosses the stop (often used to cut losses)."""
    def __init__(self, account, stock, side, qty, stop_price):
        super().__init__(account, stock, side, qty)
        self.stop_price = stop_price
    def can_fill(self, market_price):
        if self.side == Side.BUY:  return market_price >= self.stop_price
        else:                      return market_price <= self.stop_price
    def fill_price(self, market_price): return market_price


# The caller asks the order - no if/elif anywhere.
aapl = Stock("AAPL", price=180)
orders = [
    MarketOrder(alice, aapl, Side.BUY, 10),
    LimitOrder(alice, aapl, Side.BUY, 5, limit_price=170),
    StopOrder (alice, aapl, Side.SELL, 3, stop_price=175),
]

for o in orders:
    print(f"{o!r:<55} can_fill(@180)={o.can_fill(180)}  can_fill(@168)={o.can_fill(168)}")


### Why this is better 🟢

- ✅ **Open for extension, closed for modification.** Adding a `TrailingStopOrder` is a *new class* — existing code doesn't change.
- ✅ **Single Responsibility.** Each class knows its own fill rule, and nothing else.
- ✅ **Type-safe.** `Side.BUY` beats a raw string `"buy"` — typos become errors immediately.
- ✅ **Testable.** You can unit-test `LimitOrder.can_fill` without any exchange or account.

This is the pattern we'll build on in Notebook 2.


## 5️⃣ Classes at a glance

| Class | Responsibility |
|---|---|
| `Stock` | Symbol and current market price |
| `Position` | How many shares of one symbol you own + avg cost |
| `Portfolio` | `{symbol → Position}` |
| `Account` | Owns cash + a `Portfolio`, places `Order`s |
| `Order` (abstract) | Common state (side, qty, status); subclasses decide fill rules |
| `MarketOrder`, `LimitOrder`, `StopOrder` | Concrete order strategies |
| `Trade` | Record of a fill (order id, price, qty, timestamp) |
| `Exchange` | Accepts orders; matches them; produces `Trade`s |

➡️ In **Notebook 2** we turn this design into a working mini-brokerage you can run.
➡️ In **Notebook 3** we level up with an **order book**, **observers** for live P&L, and a **decorator** for risk checks.
